# Notebook 3 — Pipeline Integrado End-to-End e Validação com Great Expectations

## Aula 8: Integração e Revisão (Ferramentas e Validação)

### Objetivos

1. Implementar um **pipeline integrado end-to-end** combinando Evidently + scikit-learn + alertas.
2. Aplicar **Great Expectations** para validação de dados de entrada antes da predição.
3. Executar **re-treino automático** quando drift significativo for detectado.
4. Gerar **relatório consolidado** com métricas de drift, qualidade de dados e performance.
5. Discutir boas práticas de **MLOps** e **governança de modelos** (checklist de produção).

### Conexão com o Documento 04

> *"A integração de estratégias de detecção de drifts, validação de dados e ação rápida*
> *(re-treino/ajustes) forma o coração de um sistema de aprendizagem de máquina*
> *resiliente e escalável."*
> — DOCUMENTO_AULA_8.md, seção 'Pipelines Integrados e MLOps'

### Vídeos Relacionados

- **Vídeo 8.3**: Pipeline integrado end-to-end de detecção e mitigação.
- **Vídeo 8.4**: Validação, governança e tendências futuras.

In [ ]:
# Imports
import sys
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Adicionar diretório pai ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COL,
    DataPreprocessor,
)
from src.model import FraudDetector
from src.training import train_model, retrain_on_drift
from src.evaluation import (
    calculate_metrics,
    calculate_drift_metrics,
    calculate_all_features_drift,
    generate_classification_report,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_drift_summary,
)
from src.utils import save_model, load_model, save_metrics, set_seed

# Configurações
set_seed(42)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

%matplotlib inline

## 1. Carregamento e Preparação dos Dados

Carregamos o dataset completo e preparamos os dados de referência e produção
para executar o pipeline integrado.

In [ ]:
# Carregar e preparar dados
preprocessor = DataPreprocessor()
df = preprocessor.load_data("../data/raw/dataset.csv")

df_ref = df[df["periodo"] == "referencia"].copy()
df_prod = df[df["periodo"] == "producao"].copy()

# Pré-processar referência (fit)
df_ref_clean = preprocessor.clean_data(df_ref)
df_ref_prepared = preprocessor.prepare_features(df_ref_clean, fit=True)

# Pré-processar produção (transform)
df_prod_clean = preprocessor.clean_data(df_prod)
df_prod_prepared = preprocessor.prepare_features(df_prod_clean, fit=False)

feature_cols = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c in df_ref_prepared.columns]

X_ref = df_ref_prepared[feature_cols]
y_ref = df_ref_prepared[TARGET_COL]
X_prod = df_prod_prepared[feature_cols]
y_prod = df_prod_prepared[TARGET_COL]

# Split treino/validação apenas no referência
X_train, X_val, y_train, y_val = preprocessor.split_data(
    df_ref_prepared, test_size=0.2, random_state=42
)

print(f"Ref treino: {X_train.shape} | Ref val: {X_val.shape} | Prod: {X_prod.shape}")

## 2. Validação de Dados com Great Expectations

Conforme o **Snippet 3** do Hands On do Documento 04:
> *"É fundamental prevenir problemas garantindo que os dados atendam a regras de*
> *qualidade definidas a priori."*

Implementamos validações de schema e valores conforme as "expectativas"
do Great Expectations. Se o GX não estiver instalado, executamos validações
equivalentes manualmente.

In [ ]:
# Validação de dados com Great Expectations
# Implementa Snippet 3 do Hands On do Documento 04

validation_results = {"timestamp": datetime.utcnow().isoformat(), "checks": []}

try:
    import great_expectations as gx

    # Criar contexto GX
    context = gx.get_context()
    
    # Criar datasource e validar dados de produção
    data_source = context.data_sources.add_pandas("prod_data")
    data_asset = data_source.add_dataframe_asset("transactions")
    batch_definition = data_asset.add_batch_definition_whole_dataframe("full_batch")
    batch = batch_definition.get_batch(batch_parameters={"dataframe": df_prod})

    # Definir expectativas — conforme Snippet 3 do Doc 04
    suite = gx.ExpectationSuite(name="fraud_data_validation")
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column="transaction_id")
    )
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="valor_transacao", min_value=0, max_value=100000
        )
    )
    suite.add_expectation(
        gx.expectations.ExpectColumnMeanToBeBetween(
            column="tempo_conta_cliente", min_value=0, max_value=3650
        )
    )
    suite = context.suites.add(suite)

    # Validar
    validation_definition = context.validation_definitions.add(
        gx.ValidationDefinition(
            name="prod_validation",
            data=batch_definition,
            suite=suite,
        )
    )
    result = validation_definition.run(batch_parameters={"dataframe": df_prod})
    
    print(f"Great Expectations — Validação: {'PASSED ✓' if result.success else 'FAILED ✗'}")
    validation_results["gx_success"] = result.success

except ImportError:
    print("Great Expectations não instalado. Executando validações manuais equivalentes.")
    print("Instale com: pip install great-expectations")

except Exception as e:
    print(f"Erro ao executar GX: {e}")
    print("Executando validações manuais equivalentes.")

# Validações manuais (sempre executam como fallback/complemento)
print("\n--- Validações Manuais (equivalentes às expectativas do GX) ---")

# 1. transaction_id não nulo
null_ids = df_prod["transaction_id"].isna().sum()
check_1 = null_ids == 0
print(f"  ✓ transaction_id não nulo: {check_1} ({null_ids} nulos)")
validation_results["checks"].append({"name": "transaction_id_not_null", "passed": check_1})

# 2. valor_transacao entre 0 e 100000
val_min = df_prod["valor_transacao"].min()
val_max = df_prod["valor_transacao"].max()
check_2 = val_min >= 0 and val_max <= 100000
print(f"  ✓ valor_transacao em [0, 100000]: {check_2} (min={val_min:.2f}, max={val_max:.2f})")
validation_results["checks"].append({"name": "valor_transacao_range", "passed": bool(check_2)})

# 3. Média de tempo_conta_cliente razoável
mean_tempo = df_prod["tempo_conta_cliente"].mean()
check_3 = 0 <= mean_tempo <= 3650
print(f"  ✓ tempo_conta_cliente média razoável: {check_3} (média={mean_tempo:.1f})")
validation_results["checks"].append({"name": "tempo_conta_media", "passed": bool(check_3)})

# 4. Todas colunas obrigatórias presentes
required = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COL]
missing = set(required) - set(df_prod.columns)
check_4 = len(missing) == 0
print(f"  ✓ Schema completo: {check_4} (colunas faltantes: {missing or 'nenhuma'})")
validation_results["checks"].append({"name": "schema_complete", "passed": check_4})

all_passed = all(c["passed"] for c in validation_results["checks"])
print(f"\nResultado geral: {'TODAS VALIDAÇÕES PASSARAM ✓' if all_passed else 'FALHA EM VALIDAÇÃO ✗'}")

## 3. Pipeline Integrado: Treino → Drift → Re-treino

Conforme o Documento 04 (seção 'Pipelines Integrados e MLOps'):
> *"Inicialmente, tudo está bem — o modelo mantém a acurácia esperada. Depois de*
> *alguns meses, a etapa de monitoramento acusa drifts significativos e o NannyML*
> *estima que a acurácia caiu. Nesse momento, um alerta é enviado [...]*
> *e parte-se para o re-treino do modelo."*

Vamos simular este pipeline completo.

In [ ]:
# === ETAPA 1: Treinar modelo inicial ===
print("=" * 60)
print("ETAPA 1: Treinamento do modelo inicial")
print("=" * 60)

detector_v1 = train_model(
    X_train, y_train,
    n_estimators=100, max_depth=10,
    random_state=42, store_reference=True,
)

metrics_v1_val = calculate_metrics(
    y_val.values,
    detector_v1.predict(X_val),
    detector_v1.predict_proba(X_val)[:, 1],
)
print(f"Performance v1 (validação): AUC={metrics_v1_val['roc_auc']:.4f}, F1={metrics_v1_val['f1']:.4f}")

In [ ]:
# === ETAPA 2: Monitoramento de drift ===
print("\n" + "=" * 60)
print("ETAPA 2: Monitoramento de drift nos dados de produção")
print("=" * 60)

drift_check = detector_v1.check_drift(X_prod)
n_drifted = sum(1 for v in drift_check.values() if v["drift_detected"])
print(f"Features com drift: {n_drifted}/{len(drift_check)}")

# Performance nos dados de produção (simulando "realidade")
metrics_v1_prod = calculate_metrics(
    y_prod.values,
    detector_v1.predict(X_prod),
    detector_v1.predict_proba(X_prod)[:, 1],
)
print(f"Performance v1 (produção): AUC={metrics_v1_prod['roc_auc']:.4f}, F1={metrics_v1_prod['f1']:.4f}")

# Alerta de drift
DRIFT_THRESHOLD_FEATURES = 3
if n_drifted >= DRIFT_THRESHOLD_FEATURES:
    print(f"\n⚠️ ALERTA: {n_drifted} features com drift detectado (threshold={DRIFT_THRESHOLD_FEATURES}).")
    print("→ Acionando re-treino automático.")
    retrain_needed = True
else:
    print("✓ Drift dentro dos limites aceitáveis.")
    retrain_needed = False

In [ ]:
# === ETAPA 3: Re-treino automático ===
print("\n" + "=" * 60)
print("ETAPA 3: Re-treino automático do modelo")
print("=" * 60)

if retrain_needed:
    # Combinar dados de referência + produção para re-treino
    X_retrain = pd.concat([X_ref, X_prod], ignore_index=True)
    y_retrain = pd.concat([y_ref, y_prod], ignore_index=True)

    detector_v2 = train_model(
        X_retrain, y_retrain,
        n_estimators=100, max_depth=10,
        random_state=42, store_reference=True,
        save_path="../outputs/models/fraud_detector_v2.pkl",
    )

    # Avaliar modelo v2 nos dados de produção
    metrics_v2_prod = calculate_metrics(
        y_prod.values,
        detector_v2.predict(X_prod),
        detector_v2.predict_proba(X_prod)[:, 1],
    )
    print(f"Performance v2 (produção): AUC={metrics_v2_prod['roc_auc']:.4f}, F1={metrics_v2_prod['f1']:.4f}")

    # Comparação v1 vs v2
    print("\n--- Comparação v1 vs v2 (dados de produção) ---")
    for name in metrics_v1_prod:
        delta = metrics_v2_prod[name] - metrics_v1_prod[name]
        arrow = "↑" if delta > 0 else "↓"
        print(f"  {name}: {metrics_v1_prod[name]:.4f} → {metrics_v2_prod[name]:.4f} ({arrow} {abs(delta):.4f})")
else:
    print("Re-treino não necessário. Modelo v1 mantido.")
    detector_v2 = detector_v1
    metrics_v2_prod = metrics_v1_prod

## 4. Relatório Consolidado

Conforme Vídeo 8.3, geramos um **relatório consolidado** com:
- Métricas de drift por feature
- Qualidade de dados (validações)
- Performance do modelo (antes e após re-treino)
- Decisão tomada (re-treino ou não)

In [ ]:
# Gerar relatório consolidado
drift_df = calculate_all_features_drift(df_ref, df_prod, NUMERIC_FEATURES)

report = {
    "timestamp": datetime.utcnow().isoformat(),
    "pipeline_version": "1.0",
    "data_quality": {
        "validations_passed": all_passed,
        "checks": validation_results["checks"],
        "n_reference": len(df_ref),
        "n_production": len(df_prod),
    },
    "drift_analysis": {
        "n_features_total": len(drift_df),
        "n_features_drifted": int(drift_df["ks_drift_detected"].sum()),
        "features_drifted": list(drift_df[drift_df["ks_drift_detected"]].index),
        "threshold_features": DRIFT_THRESHOLD_FEATURES,
        "retrain_triggered": retrain_needed,
    },
    "model_performance": {
        "v1_validation": metrics_v1_val,
        "v1_production": metrics_v1_prod,
        "v2_production": metrics_v2_prod,
    },
}

# Salvar relatório
report_path = "../outputs/logs/pipeline_report.json"
save_metrics(report, report_path)

# Exibir
print("=" * 70)
print("RELATÓRIO CONSOLIDADO — Pipeline de Monitoramento de Fraude")
print("=" * 70)
print(f"\n📅 Timestamp: {report['timestamp']}")
print(f"\n📊 Qualidade de Dados:")
print(f"   Validações: {'PASSED ✓' if all_passed else 'FAILED ✗'}")
print(f"   Referência: {report['data_quality']['n_reference']} registros")
print(f"   Produção: {report['data_quality']['n_production']} registros")
print(f"\n🔍 Análise de Drift:")
print(f"   Features com drift: {report['drift_analysis']['n_features_drifted']}/{report['drift_analysis']['n_features_total']}")
print(f"   Features afetadas: {report['drift_analysis']['features_drifted']}")
print(f"   Re-treino acionado: {'SIM' if retrain_needed else 'NÃO'}")
print(f"\n🤖 Performance do Modelo:")
print(f"   v1 (validação):  AUC={metrics_v1_val['roc_auc']:.4f}")
print(f"   v1 (produção):   AUC={metrics_v1_prod['roc_auc']:.4f}")
print(f"   v2 (produção):   AUC={metrics_v2_prod['roc_auc']:.4f}")

## 5. Visualização do Pipeline

Comparação visual da performance antes e depois do re-treino.

In [ ]:
# Comparação visual v1 vs v2
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Métricas comparativas
metric_names = list(metrics_v1_prod.keys())
v1_vals = [metrics_v1_prod[m] for m in metric_names]
v2_vals = [metrics_v2_prod[m] for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35
axes[0].bar(x - width/2, v1_vals, width, label="v1 (antes)", color="#FF5722", alpha=0.8)
axes[0].bar(x + width/2, v2_vals, width, label="v2 (após re-treino)", color="#4CAF50", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names, rotation=30)
axes[0].set_ylabel("Score")
axes[0].set_title("Performance: v1 vs v2")
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3, axis="y")

# 2. Drift summary
drift_df_sorted = drift_df.sort_values("ks_statistic", ascending=True)
colors = ["#FF5722" if d else "#4CAF50" for d in drift_df_sorted["ks_drift_detected"]]
axes[1].barh(drift_df_sorted.index, drift_df_sorted["ks_statistic"], color=colors)
axes[1].set_xlabel("KS Statistic")
axes[1].set_title("Drift por Feature")
axes[1].grid(True, alpha=0.3, axis="x")

# 3. Status do pipeline
pipeline_steps = ["Validação\nDados", "Detecção\nDrift", "Re-treino", "Deploy\nv2"]
step_status = ["✓", "✓", "✓" if retrain_needed else "—", "✓" if retrain_needed else "—"]
step_colors = ["#4CAF50" if s == "✓" else "#9E9E9E" for s in step_status]

axes[2].barh(pipeline_steps, [1] * len(pipeline_steps), color=step_colors)
for i, (step, status) in enumerate(zip(pipeline_steps, step_status)):
    axes[2].text(0.5, i, status, ha="center", va="center", fontsize=18, fontweight="bold", color="white")
axes[2].set_xlim(0, 1)
axes[2].set_title("Status do Pipeline")
axes[2].set_xticks([])

plt.suptitle("Pipeline Integrado End-to-End — Relatório Visual", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/figures/03_pipeline_report.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Confusion Matrix: v1 vs v2

Comparação visual das matrizes de confusão do modelo antes e após o re-treino,
mostrando como o re-treino melhora a classificação nos dados com drift.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix v1
from sklearn.metrics import confusion_matrix
cm_v1 = confusion_matrix(y_prod, detector_v1.predict(X_prod))
sns.heatmap(cm_v1, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Legítima", "Fraude"],
            yticklabels=["Legítima", "Fraude"], ax=axes[0])
axes[0].set_title(f"v1 — F1={metrics_v1_prod['f1']:.3f}")
axes[0].set_xlabel("Predição")
axes[0].set_ylabel("Real")

# Confusion matrix v2
cm_v2 = confusion_matrix(y_prod, detector_v2.predict(X_prod))
sns.heatmap(cm_v2, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Legítima", "Fraude"],
            yticklabels=["Legítima", "Fraude"], ax=axes[1])
axes[1].set_title(f"v2 (pós re-treino) — F1={metrics_v2_prod['f1']:.3f}")
axes[1].set_xlabel("Predição")
axes[1].set_ylabel("Real")

plt.suptitle("Confusion Matrix: Antes vs Após Re-treino", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/figures/03_confusion_matrix_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Checklist de Monitoramento em Produção

Conforme Vídeo 8.4 (Validação, Governança e Tendências Futuras), o Documento 04
propõe boas práticas de MLOps e governança. Abaixo, um checklist completo
para monitoramento de modelos em produção:

| # | Item | Ferramenta/Método | Status |
|---|------|-------------------|--------|
| 1 | Validação de schema dos dados de entrada | Great Expectations | ✓ |
| 2 | Validação de ranges e valores aceitáveis | Great Expectations | ✓ |
| 3 | Detecção de covariate drift (KS, Wasserstein) | Evidently AI | ✓ |
| 4 | Estimativa de performance sem ground truth | NannyML CBPE | ✓ |
| 5 | Alertas automáticos por threshold de drift | Pipeline customizado | ✓ |
| 6 | Re-treino automático quando necessário | Pipeline customizado | ✓ |
| 7 | Versionamento de modelos (v1, v2...) | joblib + timestamped files | ✓ |
| 8 | Logging de métricas e decisões | JSON + timestamps | ✓ |
| 9 | Documentação de dados e modelo | data/README.md | ✓ |
| 10 | Testes unitários do pipeline | pytest | ✓ |

### Referências sobre Governança (Documento 04):
- **Sculley et al. (2015)**: "Hidden technical debt in machine learning systems"
- **Polyzotis et al. (2019)**: Validação contínua de dados em pipelines Google
- **Gartner (2024)**: Até 80% das organizações maduras terão SLOs para dados/modelos

In [ ]:
# Resumo final do pipeline
print("\n" + "=" * 70)
print("RESUMO FINAL DO PIPELINE INTEGRADO")
print("=" * 70)
print(f"\n1. Dados validados: {len(df_prod)} transações de produção")
print(f"2. Drift detectado em {n_drifted} features")
print(f"3. Re-treino {'executado' if retrain_needed else 'não necessário'}")
print(f"4. Modelo final: {'v2 (re-treinado)' if retrain_needed else 'v1 (original)'}")
print(f"5. AUC final: {metrics_v2_prod['roc_auc']:.4f}")
print(f"6. Relatório salvo em: outputs/logs/pipeline_report.json")
print(f"\nPipeline executado com sucesso! ✓")

## Resumo e Conclusões

### O que aprendemos neste notebook:

1. **Great Expectations**: Validação de dados com regras declarativas (schema, ranges, médias).
2. **Pipeline integrado**: Fluxo completo de validação → drift → re-treino → deploy.
3. **Re-treino automático**: Baseado no número de features com drift detectado.
4. **Relatório consolidado**: Métricas de drift, qualidade e performance em um JSON.
5. **Checklist de produção**: 10 itens essenciais para governança de modelos.

### Conceitos-chave consolidados (Aulas 1–8):

- **Covariate drift** ($P(X)$ muda): detectado via KS, Wasserstein, Evidently.
- **Concept drift** ($P(Y|X)$ muda): detectado via queda de performance, NannyML.
- **Prior drift** ($P(Y)$ muda): detectado via mudança na taxa de fraude.
- **Validação de dados**: Great Expectations como camada preventiva.
- **MLOps**: Monitoramento contínuo, re-treino, versionamento, logging.

> *"Manter um modelo relevante e confiável requer monitoramento constante e refinamento*
> *— uma lição crucial para qualquer Engenheiro de Machine Learning."*
> — DOCUMENTO_AULA_8.md, seção 'O Que Você Viu Nesta Aula'

---
*Notebook conectado com: DOCUMENTO_AULA_8.md | Vídeos 8.3 e 8.4*